# Práctica 2 · Tautomería y protonación como microespacio químico
### Del SMILES a la distribución de Boltzmann sobre microespacios de tautómeros

| Campo | Valor |
|---|---|
| **Bloque** | 1 — Generación de estructuras y espacio conformacional |
| **Nivel** | Básico |
| **Tiempo estimado** | 1.5 h (semilla) + 2 h (bosque y análisis) |
| **Pipeline** | SMILES → enumeración tautómeros → geometría 3D → GFN2-xTB → ΔG → Boltzmann → microespacio |
| **Prerequisito** | Práctica 1 completada |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/p02_tautomeria)


---
## Introducción química

Cuando un químico escribe el **SMILES** de una molécula, implícitamente
elige *una* forma tautomérica y *un* estado de protonación. Sin embargo,
en condiciones reales —por ejemplo en solución, en el sitio activo de una enzima, etc—
la misma molécula puede existir simultáneamente en **múltiples formas
que se interconvierten rápidamente**: se habla de *especiación química*.

Esta especiación no es un artefacto computacional: tiene consecuencias
directas sobre la reactividad, la solubilidad y la afinidad de unión
a proteínas ([Abrahamsson, 2024](https://doi.org/10.1039/d4ra06695b)).  Si el cálculo se ejecuta sobre el tautómero
incorrecto, todos los resultados subsecuentes son los de una
especie que puede ser minoritaria o inexistente.

### ¿Qué haremos en esta práctica?

```
SMILES semilla
     │
     ▼
 Enumeración de tautómeros/protómeros
     │            (RDKit TautomerEnumerator)
     ▼
 Geometría 3D para cada forma
     │            (ETKDG v3 + MMFF94)
     ▼
 Optimización + frecuencias
     │            (GFN2-xTB, --opt tight --ohess)
     ▼
 ΔG relativo entre formas
     │            (1 Hartree = 2625.5 kJ/mol)
     ▼
 Distribución de Boltzmann  →  microespacio químico
```

> **🌱 Semilla:** 2-piridona / 2-hidroxipiridina — el par tautómero clásico
> de la química heterocíclica, con valor experimental de referencia.
>
> **🌳 Bosque:** 40 moléculas con tautomería documentada (bases nucleicas,
> fármacos, heterociclos, sistemas ceto-enol).


---
## Cuadro conceptual: los tres lenguajes lineales de moléculas

Antes de escribir código, necesitamos entender cómo se representan
las moléculas como texto. Hay tres sistemas que aparecen en esta práctica:

---

### SMILES — *Simplified Molecular Input Line Entry System*

**Qué es:** Una cadena de texto que codifica la *topología* de una molécula
(qué átomos hay y cómo están conectados) usando letras, dígitos y símbolos.

| Fragmento SMILES | Significado |
|---|---|
| `C`, `N`, `O` | carbono, nitrógeno, oxígeno (sp3 implícito) |
| `c`, `n`, `o` | átomos aromáticos (en minúscula) |
| `=`, `#` | enlace doble, triple |
| `()` | ramificación |
| `1`, `2`... | cierre de anillo |
| `[H]`, `[NH+]` | hidrógenos o cargas explícitos |

**Ejemplo clave de esta práctica:**
```
O=c1ccccn1H   →  2-piridona   (forma ceto/lactama)
Oc1ccccn1     →  2-hidroxipiridina  (forma enol)
```

> ⚠️ **Limitación importante:** Un mismo SMILES puede representar distintos
> tautómeros. `O=c1ccccn1H` y `Oc1ccccn1` son el *mismo compuesto* pero
> **formas tautómeras distintas**. El software necesita decidir cuál usar;
> esto es exactamente el problema que resuelve esta práctica.

---

### SMARTS — *SMILES Arbitrary Target Specification*

**Qué es:** Un lenguaje de **patrones** o expresiones regulares para moléculas.
Mientras SMILES describe *una* molécula específica, SMARTS describe
*una clase* de subestructuras moleculares.

**¿Por qué importa aquí?** El algoritmo `TautomerEnumerator` de RDKit funciona
internamente con reglas SMARTS: define los **patrones de transformación tautómera**
como pares de SMARTS (patrón reactivo → patrón producto).

Por ejemplo, una regla de tautomería ceto-enol puede codificarse como:
```
[CX4:1][C:2]=[O:3]  >>  [C:1]=[C:2][OH:3]
```
Esto dice: "si ves un carbono sp3 (CX4) unido a un C=O, genera también
la forma enólica C=C–OH".

| Símbolo SMARTS | Significado |
|---|---|
| `[CX4]` | carbono con 4 vecinos (sp3) |
| `[#6]` | cualquier isótopo de carbono |
| `[!H]` | átomo sin hidrógenos implícitos |
| `:1`, `:2` | etiqueta de átomo (mapeo de reacción) |

> **Ventaja:** SMARTS permite buscar *cualquier* molécula que tenga ese patrón.
> Es la base de todas las búsquedas de subestructura en quimioinformática.
>
> **Limitación:** Si un tipo de tautomería no está en las reglas de Sitzmann
> que usa RDKit, **el enumerador no lo generará**. Esto es el núcleo de la
> Pregunta de Discusión 6.

---

### SELFIES — *Self-Referencing Embedded Strings*

**Qué es:** Un sistema de representación lineal diseñado en 2020 para
**garantizar que toda cadena válida corresponde a una molécula química válida**.
A diferencia de SMILES, donde una cadena aleatoria casi seguro es inválida,
*cualquier* cadena SELFIES bien formada es decodificable.

**¿Por qué esto importa?** En modelos de aprendizaje automático generativo
(VAEs, diffusion models), si el modelo aprende SMILES puede generar strings
inválidos. SELFIES elimina ese problema por diseño.

```python
# SELFIES del acetileno (comparación):
SMILES:  C#C
SELFIES: [C][=C][H]   # (aproximado; depende de la versión)
```

> **En esta práctica** usamos SMILES (más legibles para humanos y bien
> soportados por RDKit). SELFIES aparece en el Bloque 5 (modelos generativos).

---

**📌 Espacio para diagrama** *(insertar aquí)*:
Un esquema visual comparando los tres sistemas con el mismo ejemplo
(p.ej. la 2-piridona): SMILES → estructura 2D → SMARTS que la captura → SELFIES.
Sugerencia: diagrama de cuatro cuadrantes con flechas y código de colores.


---
## Marco teórico esencial

### 1. El equilibrio tautomérico

Dos moléculas son **tautómeros** si se interconvierten por migración de
un protón con desplazamiento concomitante de un enlace doble. El equilibrio
entre T₁ y T₂ está gobernado por:

$$K_T = \frac{[T_2]}{[T_1]} = \exp\!\left(-\frac{\Delta G^\circ_{T_1 \to T_2}}{RT}\right)$$

A 298 K, **5.7 kJ/mol** implica que la forma minoritaria representa ~10%.
**17 kJ/mol** la reduce al 0.1%.

### 2. Distribución de Boltzmann

Para $n$ tautómeros con energías libres $G_i$:

$$p_i = \frac{e^{-G_i/RT}}{\sum_{j=1}^{n} e^{-G_j/RT}}$$

Esta expresión convierte un conjunto de energías absolutas en una
**distribución de probabilidad**: $p_i$ es la fracción de moléculas
en la forma $i$ en el equilibrio.

### 3. GFN2-xTB y la energía libre de Gibbs

Con la bandera `--ohess`, xTB calcula:

$$G = E_{\text{elec}} + E_{\text{ZPE}} + H_{\text{vib}}(T) - TS_{\text{vib}}(T) + H_{\text{rot}} + H_{\text{trans}} - TS_{\text{rot}} - TS_{\text{trans}}$$

Para la *comparación entre tautómeros*, las contribuciones traslacionales
y rotacionales se cancelan en gran medida. El resultado es dominado por:
$\Delta E_{\text{elec}} + \Delta$ZPE.

> **Validación de referencia:** Beak (1976) reportó
> $\Delta G^\circ_{\text{gas}} \approx -6.3$ kJ/mol a favor de la 2-piridona.
> Abrahamsson (2024) validó GFN2-xTB para este tipo de problemas.
> **Error típico: < 2 kJ/mol** para tautómeros de heterociclos.


---
## ⚙️ Configuración del entorno

**Ejecuta esta celda primero**, antes de cualquier otra.
Instala las dependencias necesarias (solo en Colab; en local ya deberías
tener el entorno `qc-manual` activado de la Práctica 1).


In [ ]:
# ── Instalación (solo en Google Colab) ──────────────────────────────────────
# En entorno local con conda: conda activate qc-manual
import sys, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run(['pip', 'install', '-q', 'rdkit', 'py3Dmol',
                    'plotly', 'kaleido'], check=False)

# ── Imports centrales ─────────────────────────────────────────────────────────
# Todos los imports de la práctica van aquí; no se repiten en celdas posteriores.
import os, re, subprocess, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
warnings.filterwarnings('ignore')

# RDKit — quimioinformática
try:
    from rdkit import Chem
    from rdkit.Chem import AllChem, Draw, rdmolfiles
    from rdkit.Chem.MolStandardize import rdMolStandardize
    from rdkit.Chem import rdMolDescriptors
    print('✓ RDKit', Chem.__version__ if hasattr(Chem,'__version__') else 'OK')
    RDKIT_OK = True
except ImportError:
    print('✗ RDKit no disponible — instala: conda install -c conda-forge rdkit')
    RDKIT_OK = False

# py3Dmol — visualización 3D interactiva en el notebook
try:
    import py3Dmol
    print('✓ py3Dmol OK')
    PY3DMOL_OK = True
except ImportError:
    print('⚠ py3Dmol no disponible — las celdas 3D mostrarán instrucciones de Avogadro2')
    PY3DMOL_OK = False

# Plotly — visualizaciones interactivas (tableros HTML)
try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    print('✓ Plotly OK')
    PLOTLY_OK = True
except ImportError:
    print('⚠ Plotly no disponible — instala: pip install plotly')
    PLOTLY_OK = False

# xtb — verificar disponibilidad en PATH
xtb_check = subprocess.run(['which', 'xtb'], capture_output=True, text=True)
XTB_OK = xtb_check.returncode == 0
print('✓ xtb:', xtb_check.stdout.strip() if XTB_OK else
      '✗ no encontrado — se usarán energías simuladas para el fallback')

# ── Constantes globales ───────────────────────────────────────────────────────
R   = 8.314e-3   # kJ / (mol·K)  — constante de los gases
T   = 298.15     # K             — temperatura estándar
EH2KJ = 2625.5   # kJ/mol por Hartree (factor de conversión de unidades xTB)

print('\n✓ Configuración lista. Constantes: R =', R, 'kJ/(mol·K), T =', T, 'K')


---
## 📝 Preguntas previas

Responde **antes de ejecutar cualquier celda de cálculo**.
Tus respuestas iniciales quedan registradas en el notebook;
las revisarás al final comparando con los resultados.


**Pregunta 1 (Conceptual).** La guanina tiene cuatro tautómeros
relevantes biológicamente: ceto-amino (dominante), enol-amino, ceto-imino
y enol-imino. Sin calcular nada, ¿cuál esperas que sea el más estable
y por qué? ¿Qué consecuencia tendría para el apareamiento de bases
si el tautómero minoritario estuviera presente en el 0.01% de las moléculas?


In [ ]:
respuesta_1 = """
Escribe aquí tu respuesta a la Pregunta 1.
"""
print(respuesta_1)


**Pregunta 2 (predictivo).** Para la 2-piridona hay dos formas:
la forma **ceto** (lactama: C=O y N-H) y la forma **enol**
(2-hidroxipiridina: C-OH, N sin H).
¿Cuál esperas que sea más estable en fase gas?
¿Cambiaría ese orden en agua? ¿Por qué?



In [ ]:
respuesta_2 = """
Escribe aquí tu respuesta a la Pregunta 2.
"""
print(respuesta_2)


**Pregunta 3 (Crítica).** El pipeline de esta práctica calcula energías en vacío (fase gas)
y sobre esas energías se aplica la distribución de Boltzmann. ¿En qué circunstancias daría
predicciones incorrectas sobre cuál tautómero predomina *en solución acuosa*?
¿Qué habría que añadir al pipeline para corregirlo?


In [ ]:
respuesta_3 = (
    'El protocolo en vacío fallaría cuando...\n'
    'Para corregirlo habría que añadir...'
)
print(respuesta_3)


---
## 🌱 Semilla · Paso 1 — Enumeración de tautómeros

### ¿Qué hace este bloque?

Dado el **SMILES** de la 2-piridona, el `TautomerEnumerator` de RDKit
busca **sistemáticamente** todas las formas que puede adoptar la molécula
moviendo un protón.

Funciona así internamente:
1. Lee las reglas de Sitzmann (codificadas como **patrones SMARTS**).
2. Identifica qué patrones están presentes en la molécula de entrada.
3. Aplica las transformaciones SMARTS válidas de forma combinatoria.
4. Elimina duplicados usando SMILES canónicos (dos strings distintos
   pueden representar la misma molécula).

> **Parámetro `SetMaxTautomers(100)`:** por defecto RDKit limita a 1000.
> Para moléculas con muchos grupos tautomerizables (p.ej. proteínas) el número
> crece exponencialmente. Para sistemas farmacéuticos típicos, 100 es suficiente.

**📌 Espacio para figura** *(insertar aquí)*:
Diagrama mostrando los dos tautómeros de la 2-piridona con sus estructuras
Lewis, el protón que migra resaltado en rojo, y las longitudes de enlace
C=O/C–OH anotadas. Ideal si incluyes la numeración de átomos IUPAC.


In [ ]:
# ── Definir la molécula semilla ──────────────────────────────────────────────
# SMILES de la 2-piridona: forma lactámica (ceto)
# El N lleva el H explícito porque en esta forma es el N–H el que define
# la tautomería. Sin él, RDKit podría canonicalizar a otra forma.
SMILES_SEMILLA = 'O=c1ccccn1H'   # 2-piridona (lactama, forma ceto)
NOMBRE        = '2-piridona'

mol = Chem.MolFromSmiles(SMILES_SEMILLA)
if mol is None:
    raise ValueError(f'SMILES no válido: {SMILES_SEMILLA}')

# ── Configurar el enumerador de tautómeros ────────────────────────────────────
# TautomerEnumerator usa las reglas de Sitzmann (artículo original: 2010)
# Están codificadas internamente como transformaciones SMARTS.
enumerador = rdMolStandardize.TautomerEnumerator()
enumerador.SetMaxTautomers(100)      # límite de seguridad
enumerador.SetRemoveSp3Stereo(True)  # ignorar estereoquímica sp3 durante enumeración

# ── Enumerar ──────────────────────────────────────────────────────────────────
# Enumerate() devuelve un objeto iterable de moléculas RDKit (no SMILES).
# Cada elemento es un objeto Mol con los átomos/enlaces de ese tautómero.
tautomeros = list(enumerador.Enumerate(mol))
print(f'Tautómeros encontrados para {NOMBRE}: {len(tautomeros)}')
print()

# ── Mostrar SMILES canónicos de cada tautómero ────────────────────────────────
# MolToSmiles() devuelve el SMILES canónico (único) de RDKit para esa molécula.
# Esto nos permite comparar formas aunque partan de SMILES escritos diferente.
for i, tau in enumerate(tautomeros):
    smi = Chem.MolToSmiles(tau)
    peso = rdMolDescriptors.CalcExactMolWt(tau)
    print(f'  T{i+1}: {smi}   [MW = {peso:.3f} Da]')

# ── Rejilla 2D ────────────────────────────────────────────────────────────────
# MolsToGridImage() renderiza múltiples moléculas en una cuadrícula PNG.
# Muy útil para visualizar diferencias estructurales a simple vista.
os.makedirs(f'{NOMBRE}_tautomeros', exist_ok=True)
img = Draw.MolsToGridImage(
    tautomeros,
    molsPerRow=4,
    subImgSize=(320, 260),
    legends=[f'T{i+1}: {Chem.MolToSmiles(t)}' for i, t in enumerate(tautomeros)]
)
img.save(f'{NOMBRE}_tautomeros_2D.png')
print(f'\nRejilla guardada: {NOMBRE}_tautomeros_2D.png')
display(img)


### 🔍 Demo: SMARTS en acción

Antes de continuar, veamos directamente cómo funciona SMARTS:
¿puede detectar el patrón ceto-enol en nuestra molécula?

Esto ilustra por qué `TautomerEnumerator` *sabe* que nuestra molécula
puede tautomerizarse: reconoce el patrón SMARTS apropiado.


In [ ]:
# ── SMARTS: búsqueda de subestructura ────────────────────────────────────────
# Un patrón SMARTS es como una 'expresión regular' para moléculas.
# HasSubstructMatch() devuelve True si la molécula contiene ese patrón.

# Patrón para el fragmento lactámico (C=O unido a N en anillo)
# [NX3] = nitrógeno con 3 vecinos (sp3 o sp2 con par libre)
# [CX3]=O = carbono con 3 vecinos y doble enlace a oxígeno (carbonilo)
# [r] = en un anillo
patrones = {
    'Carbonilo en anillo (lactama)' : '[NX3][CX3]=O',
    'Enol genérico'                 : '[CX3]=[CX3][OH]',
    'N–H en anillo aromático'       : '[nH]',
    'Heterociclo N piridínico'      : 'n1ccccc1',
}

print('=== Búsquedas SMARTS en los tautómeros de la 2-piridona ===\n')
print(f'{"Patrón SMARTS":<40}', '  '.join([f'T{i+1}' for i in range(len(tautomeros))]))
print('-' * 70)

for nombre_p, smarts_str in patrones.items():
    patron = Chem.MolFromSmarts(smarts_str)
    if patron is None:
        print(f'  [SMARTS inválido: {smarts_str}]')
        continue
    resultados = ['✓' if t.HasSubstructMatch(patron) else '✗' for t in tautomeros]
    print(f'{nombre_p:<40}', '   '.join(resultados))

print()
print('💡 Los distintos patrones presentes en T1 vs T2 confirman que son')
print('   formas tautómeras genuinas (no duplicados).')


---
## 🌱 Semilla · Paso 2 — Geometrías 3D para cada tautómero

### ¿Qué hace este bloque?

Para cada tautómero (que hasta ahora es solo un *grafo 2D*), necesitamos
coordenadas cartesianas 3D para poder optimizar con xTB.

El proceso tiene dos pasos:
1. **Incrustación (embedding) con ETKDG v3:** genera coordenadas 3D
   iniciales usando distancias de enlace/ángulos estándar y restricciones
   de distancia. El `randomSeed=42` garantiza reproducibilidad.
2. **Pre-optimización con MMFF94:** ajusta las coordenadas con el campo
   de fuerzas MMFF94 (o UFF como fallback). Esto reduce la energía de
   tensión antes de pasarlo a xTB, ahorrando ciclos de optimización.

> **¿Por qué pre-optimizar con MM antes de xTB?**
> xTB es más costoso que un campo de fuerzas pero más preciso.
> Empezar desde una geometría razonable acelera la convergencia
> y evita que xTB 'quede atrapado' en mínimos espurios.


In [ ]:
# ── Generar geometrías 3D para cada tautómero ────────────────────────────────
# Necesitamos un archivo XYZ por tautómero para pasarlo a xtb.
# XYZ es el formato de texto más simple: una línea por átomo con símbolo + x,y,z.

xyz_generados = []  # lista de rutas a archivos generados exitosamente

for i, tau in enumerate(tautomeros):
    etiqueta = f'{NOMBRE}_T{i+1:02d}'

    # AddHs() añade hidrógenos explícitos (necesarios para xTB)
    # Los hidrógenos implícitos en RDKit son solo contadores; xTB necesita posiciones
    tau_h = Chem.AddHs(tau)

    # ETKDGv3: "Experimental Torsion Knowledge Distance Geometry v3"
    # Usa datos estadísticos de torsiones observadas en estructuras de cristal.
    params = AllChem.ETKDGv3()
    params.randomSeed = 42  # reproducible
    ok = AllChem.EmbedMolecule(tau_h, params)

    if ok == -1:
        print(f'  {etiqueta}: ⚠ fallo en incrustación ETKDG, saltando.')
        continue

    # Pre-optimización MMFF94 (Merck Molecular Force Field)
    # Devuelve None si la molécula tiene átomos sin parámetros MMFF (p.ej. metales)
    ff = AllChem.MMFFGetMoleculeForceField(
             tau_h, AllChem.MMFFGetMoleculeProperties(tau_h))
    if ff is None:
        # UFF (Universal Force Field): parámetros para casi todos los elementos
        ff = AllChem.UFFGetMoleculeForceField(tau_h)
    if ff:
        ff.Minimize(maxIts=2000)  # máximo 2000 ciclos de minimización

    # Guardar en formato XYZ
    xyz_path = os.path.join(f'{NOMBRE}_tautomeros', f'{etiqueta}_FF.xyz')
    rdmolfiles.MolToXYZFile(tau_h, xyz_path)
    xyz_generados.append(xyz_path)
    print(f'  {etiqueta}: ✓ geometría FF guardada ({tau_h.GetNumAtoms()} átomos)')

print(f'\nTotal: {len(xyz_generados)} de {len(tautomeros)} geometrías generadas.')


---
## 🌱 Semilla · Paso 3 — Optimización GFN2-xTB

### Flags del comando xTB explicadas

```bash
xtb T01_FF.xyz --opt tight --ohess --chrg 0 --uhf 0 --gfn 2 --T 298.15
```

| Flag | Significado |
|---|---|
| `--opt tight` | Criterio de convergencia estricto: grad RMS < 2×10⁻⁴ Eh/Bohr |
| `--ohess` | Calcula frecuencias **en la geometría optimizada** (un solo paso) |
| `--chrg 0` | Carga total = 0 (neutro) |
| `--uhf 0` | Cero electrones desapareados (sistema de capa cerrada) |
| `--gfn 2` | Usa el hamiltoniano GFN2-xTB (el más preciso de la familia GFN) |
| `--T 298.15` | Temperatura para el cálculo de G(T) |

> **¿Por qué `--ohess` y no `--hess`?**
> `--hess` calcula frecuencias en la *geometría de entrada* (puede no ser mínimo).
> `--ohess` = optimización + hessiano en la geometría final. Más fiable y
> más eficiente si vas a hacer ambas cosas.

> **⚠️ Frecuencia imaginaria:** Si ves un valor `ν < 0` en el log, la geometría
> es un **punto de silla**, no un mínimo. Solución: perturbar el H y reoptimizar.


In [ ]:
# ── Optimización con GFN2-xTB ────────────────────────────────────────────────
# Si xtb no está disponible, usamos logs simulados con energías realistas
# (basadas en Beak 1976 + Abrahamsson 2024).

DIRECTORIO = f'{NOMBRE}_tautomeros'

if XTB_OK:
    print('xtb disponible — ejecutando optimizaciones reales...\n')
    for xyz_path in sorted(os.listdir(DIRECTORIO)):
        if not xyz_path.endswith('_FF.xyz'):
            continue
        base    = xyz_path.replace('_FF.xyz', '')
        log_out = os.path.join(DIRECTORIO, f'{base}_xtb.out')

        result = subprocess.run(
            ['xtb', xyz_path, '--opt', 'tight', '--ohess',
             '--chrg', '0', '--uhf', '0', '--gfn', '2', '--T', '298.15'],
            capture_output=True, text=True,
            cwd=DIRECTORIO
        )
        with open(log_out, 'w') as f:
            f.write(result.stdout)

        # Mover geometría optimizada
        opt_src = os.path.join(DIRECTORIO, 'xtbopt.xyz')
        opt_dst = os.path.join(DIRECTORIO, f'{base}_opt.xyz')
        if os.path.exists(opt_src):
            os.rename(opt_src, opt_dst)

        convergio = 'GEOMETRY OPTIMIZATION CONVERGED' in result.stdout
        print(f'  {base}: {"✓ convergió" if convergio else "⚠ no convergió"}')

else:
    print('xtb no disponible — generando logs simulados con valores de referencia.\n')
    print('Los valores de ΔG simulados son consistentes con Beak (1976):')
    print('  T1 (2-piridona):        G_sim = -19.887 Eh  [forma dominante]')
    print('  T2 (2-hidroxipiridina): G_sim = -19.885 Eh  [+6.3 kJ/mol]\n')

    # Energías en Hartree consistentes con ΔG ≈ -6.3 kJ/mol
    # -6.3 kJ/mol / 2625.5 = -0.002400 Eh
    sims = [
        ('2-piridona_T01', -19.887300, 'GEOMETRY OPTIMIZATION CONVERGED'),
        ('2-piridona_T02', -19.884900, 'GEOMETRY OPTIMIZATION CONVERGED'),
    ]
    for base, g_val, conv_msg in sims:
        log_content = (
            f'GFN2-xTB (SIMULADO — fallback pedagógico)\n'
            f'Molécula: {base}\n'
            f'{conv_msg}\n'
            f'TOTAL ENERGY      {g_val + 0.002:.6f} Eh\n'
            f'TOTAL FREE ENERGY {g_val:.6f} Eh\n'
            f'Frequencies (cm-1): 112.3 218.7 345.1 ... (todas positivas)\n'
        )
        log_path = os.path.join(DIRECTORIO, f'{base}_xtb.out')
        with open(log_path, 'w') as f:
            f.write(log_content)
        # Copiar FF.xyz como proxy de geometría optimizada
        ff_src = os.path.join(DIRECTORIO, f'{base}_FF.xyz')
        opt_dst = os.path.join(DIRECTORIO, f'{base}_opt.xyz')
        if os.path.exists(ff_src):
            import shutil; shutil.copy(ff_src, opt_dst)
        print(f'  {base}: log simulado guardado.')

print('\n✓ Paso 3 completado.')


---
## 🌱 Semilla · Paso 4 — Extracción de G(298 K) de los logs

### ¿Por qué extraemos con expresiones regulares?

Los logs de xTB son archivos de texto con un formato estable.
Usamos **`re` (módulo de expresiones regulares de Python)**
para buscar exactamente las líneas que contienen los valores que necesitamos.

La línea clave en el log tiene este formato:
```
TOTAL FREE ENERGY          -19.887300  Eh
```

El patrón `r'TOTAL FREE ENERGY\s+([-\d.]+)\s+Eh'` captura el número flotante.

> **¿Energía total vs G(298 K)?** La `TOTAL ENERGY` es $E_{\text{elec}}$.
> `TOTAL FREE ENERGY` incluye ZPE + correcciones vibracional/rotacional/traslacional
> a 298 K. Para comparar tautómeros, **siempre** usar `TOTAL FREE ENERGY`.


In [ ]:
# ── Extraer energías de Gibbs de los logs de xTB ─────────────────────────────
# re.search() busca la primera ocurrencia del patrón en el texto.
# El grupo de captura ([-\d.]+) extrae el número (incluyendo el signo negativo).

registros = []

for archivo in sorted(os.listdir(DIRECTORIO)):
    if not archivo.endswith('_xtb.out'):
        continue
    etiqueta = archivo.replace('_xtb.out', '')

    with open(os.path.join(DIRECTORIO, archivo)) as f:
        texto = f.read()

    # Energía electrónica total (sin correcciones térmicas)
    m_etot  = re.search(r'TOTAL ENERGY\s+([-\d.]+)\s+Eh', texto)
    # Energía libre de Gibbs a 298 K (incluye ZPE + correcciones)
    m_gibbs = re.search(r'TOTAL FREE ENERGY\s+([-\d.]+)\s+Eh', texto)
    # Verificar que la optimización geométrica convergió
    convergio = 'GEOMETRY OPTIMIZATION CONVERGED' in texto

    # Verificar frecuencias imaginarias (un mínimo real no debe tenerlas)
    # Una frecuencia negativa en el log indica punto de silla, no mínimo
    frec_imaginarias = len(re.findall(r'-\d+\.\d+\s+cm', texto))

    registros.append({
        'etiqueta'        : etiqueta,
        'convergio'       : convergio,
        'frec_imaginarias': frec_imaginarias,
        'E_total_Eh'      : float(m_etot.group(1))  if m_etot  else None,
        'G_298_Eh'        : float(m_gibbs.group(1)) if m_gibbs else None,
    })

df_semilla = pd.DataFrame(registros)

# Calcular ΔG respecto al tautómero más estable (mínimo de G)
# Conversión: 1 Hartree = 2625.5 kJ/mol
G_min = df_semilla['G_298_Eh'].min()
df_semilla['dG_kJ_mol'] = (df_semilla['G_298_Eh'] - G_min) * EH2KJ

# ── Informe de calidad ────────────────────────────────────────────────────────
print('=== Control de calidad ===')
for _, row in df_semilla.iterrows():
    estado_conv  = '✓' if row['convergio'] else '⚠ NO CONVERGIÓ'
    estado_frec  = '✓' if row['frec_imaginarias'] == 0 else f'⚠ {row["frec_imaginarias"]} imag.'
    print(f"  {row['etiqueta']}: convergencia={estado_conv}  frec_imag={estado_frec}")

print('\n=== Energías y ΔG ===')
print(df_semilla[['etiqueta', 'G_298_Eh', 'dG_kJ_mol']].round(6).to_string(index=False))


---
## 🌱 Semilla · Paso 5 — Distribución de Boltzmann y validación

### De energías a probabilidades

La distribución de Boltzmann conecta nuestros números computacionales
con la fracción molar de cada tautómero:

$$p_i = \frac{e^{-\Delta G_i / RT}}{\sum_j e^{-\Delta G_j / RT}}$$

Nota que trabajamos con **ΔG relativo** (no con G absoluto), porque
las constantes se cancelan. Esto es numéricamente más estable.

### Validación vs experimento

| Valor | Referencia (Beak 1976) | Este cálculo |
|---|---|---|
| ΔG°_gas (kJ/mol) | −6.3 | *(ver abajo)* |
| p(2-piridona) en gas (%) | ~93 | *(ver abajo)* |

Criterio de aceptación: error en ΔG° < 3 kJ/mol respecto al experimental.


In [ ]:
# ── Distribución de Boltzmann ─────────────────────────────────────────────────
# R = 8.314e-3 kJ/(mol·K), T = 298.15 K (definidos en la Celda 0)

# Calcular pesos de Boltzmann: exp(-ΔG_i / RT)
# Para el mínimo (ΔG=0): peso=1. Para los demás: peso<1.
df_semilla['peso_Boltzmann'] = np.exp(-df_semilla['dG_kJ_mol'] / (R * T))
suma = df_semilla['peso_Boltzmann'].sum()

# Normalizar para obtener fracciones (suma = 1 = 100%)
df_semilla['poblacion']  = df_semilla['peso_Boltzmann'] / suma
df_semilla['porcentaje'] = df_semilla['poblacion'] * 100

# Ordenar de más a menos estable
df_semilla = df_semilla.sort_values('dG_kJ_mol').reset_index(drop=True)

print('=== Distribución de Boltzmann a 298 K ===')
print(df_semilla[['etiqueta', 'dG_kJ_mol', 'porcentaje']].round(3).to_string(index=False))

# Identificar el tautómero dominante
T1 = df_semilla.iloc[0]
print(f'\n→ Tautómero dominante: {T1["etiqueta"]}')
print(f'  Población: {T1["porcentaje"]:.1f}%')

# ── Validación vs Beak (1976) ─────────────────────────────────────────────────
dG_calculado    = df_semilla['dG_kJ_mol'].iloc[1] if len(df_semilla) > 1 else 0.0
dG_experimental = 6.3   # kJ/mol (magnitud; T1 es más estable → dG_T2 > 0)
error = abs(dG_calculado - dG_experimental)

print(f'\n=== Validación ===')
print(f'  ΔG calculado (T1→T2) : {dG_calculado:.2f} kJ/mol')
print(f'  ΔG experimental (gas): {dG_experimental:.1f} kJ/mol  [Beak 1976]')
print(f'  Error absoluto       : {error:.2f} kJ/mol')
print(f'  ¿Dentro del criterio (<3 kJ/mol)? {"✓ SÍ" if error < 3 else "✗ NO — revisar"}')

# Guardar resultados de la semilla
df_semilla.to_csv(f'{NOMBRE}_tautomeros_resultados.csv', index=False)
print(f'\nResultados guardados: {NOMBRE}_tautomeros_resultados.csv')


---
## 🌱 Semilla · Paso 6 — Visualización 3D comparativa

Comparamos la geometría 3D de T1 (2-piridona) y T2 (2-hidroxipiridina)
para confirmar visualmente que son formas estructuralmente distintas.

Observa especialmente:
- La posición del **hidrógeno móvil** (en N o en O)
- La diferencia en la **longitud C=O vs C–OH**


In [ ]:
# ── Visualización 3D con py3Dmol ─────────────────────────────────────────────
# py3Dmol es una librería que permite visualizar moléculas 3D en el notebook
# usando el motor de renderizado 3Dmol.js (WebGL). Muy útil para comparaciones.

archivos_opt = sorted([
    f for f in os.listdir(DIRECTORIO) if f.endswith('_opt.xyz')
])

if PY3DMOL_OK and len(archivos_opt) >= 2:
    # Crear viewer con dos paneles lado a lado
    viewer = py3Dmol.view(width=700, height=350, linked=True, viewergrid=(1, 2))

    estilos = [
        {'stick': {'colorscheme': 'greenCarbon', 'radius': 0.15},
         'sphere': {'scale': 0.25}},
        {'stick': {'colorscheme': 'orangeCarbon', 'radius': 0.15},
         'sphere': {'scale': 0.25}},
    ]

    for idx, (arch, estilo) in enumerate(zip(archivos_opt[:2], estilos)):
        xyz_path = os.path.join(DIRECTORIO, arch)
        with open(xyz_path) as f:
            xyz_content = f.read()
        etiq = arch.replace('_opt.xyz', '')
        viewer.addModel(xyz_content, 'xyz', viewer=(0, idx))
        viewer.setStyle(estilo, viewer=(0, idx))
        viewer.addLabel(etiq, {'position': {'x': 0, 'y': 2.5, 'z': 0},
                                'fontSize': 14, 'fontColor': 'black'},
                        viewer=(0, idx))
        viewer.zoomTo(viewer=(0, idx))

    viewer.show()
    print('Verde: T1 (2-piridona)  |  Naranja: T2 (2-hidroxipiridina)')
    print('Arrastra con el ratón para rotar. Usa la rueda para zoom.')

else:
    print('py3Dmol no disponible o faltan geometrías optimizadas.')
    print('Para visualizar en 3D:')
    print('  1. Instala Avogadro2: https://two.avogadro.cc')
    print('  2. Abre los archivos *_opt.xyz de la carpeta', DIRECTORIO)
    print('  3. Compara T01_opt.xyz vs T02_opt.xyz')


---
## 🌱 Semilla · Paso 7 — Validación geométrica

Más allá de la energía, podemos confirmar la identidad de cada
tautómero midiendo **distancias de enlace clave**:

| Enlace | 2-piridona (ceto) | 2-hidroxipiridina (enol) |
|---|---|---|
| C=O / C–O | ~1.22 Å | ~1.34 Å |
| O–H | no existe | ~0.97 Å |
| N–H | ~1.01 Å | no existe |

Estas diferencias son lo suficientemente grandes como para ser
inequívocas en la geometría optimizada.


In [ ]:
# ── Validación geométrica: distancias C=O y C–OH ─────────────────────────────
# Cargamos las geometrías optimizadas con RDKit y medimos las distancias
# que distinguen inequívocamente los dos tautómeros.

from rdkit.Chem import rdMolTransforms

print('=== Validación geométrica de distancias de enlace ===')
print()

for arch in sorted(os.listdir(DIRECTORIO)):
    if not arch.endswith('_opt.xyz'):
        continue

    xyz_path = os.path.join(DIRECTORIO, arch)
    mol_opt  = rdmolfiles.MolFromXYZFile(xyz_path)

    if mol_opt is None:
        print(f'  {arch}: no se pudo cargar con RDKit (normal para XYZ de xTB)')
        # Alternativa: leer XYZ manualmente y calcular distancias
        with open(xyz_path) as f:
            lines = f.readlines()[2:]  # saltar cabecera XYZ
        atomos = []
        for line in lines:
            parts = line.split()
            if len(parts) == 4:
                atomos.append((parts[0], float(parts[1]),
                                float(parts[2]), float(parts[3])))
        # Calcular distancias O-H y N-H
        oxigenos  = [(i,a) for i,a in enumerate(atomos) if a[0]=='O']
        nitrogenos= [(i,a) for i,a in enumerate(atomos) if a[0]=='N']
        hidrogenos= [(i,a) for i,a in enumerate(atomos) if a[0]=='H']

        print(f'  {arch.replace("_opt.xyz","")}: {len(atomos)} átomos en XYZ')
        for oi, o in oxigenos:
            for hi, h in hidrogenos:
                d = np.sqrt((o[1]-h[1])**2 + (o[2]-h[2])**2 + (o[3]-h[3])**2)
                if d < 1.2:  # distancia O–H típica
                    print(f'    O–H detectado: {d:.3f} Å → confirma forma ENOL (C–OH)')
        for ni, n in nitrogenos:
            for hi, h in hidrogenos:
                d = np.sqrt((n[1]-h[1])**2 + (n[2]-h[2])**2 + (n[3]-h[3])**2)
                if d < 1.15:  # distancia N–H típica
                    print(f'    N–H detectado: {d:.3f} Å → confirma forma CETO (lactama)')
        print()


---
## 🌳 Parte 2 · El bosque — 40 moléculas con tautomería documentada

### Estrategia de expansión

El bosque contiene 40 moléculas elegidas para representar la diversidad
de la tautomería en contextos reales:

| Familia | N | Ejemplos |
|---|---|---|
| Bases nucleicas | 8 | adenina, guanina, citosina, uracilo, timina, hipoxantina |
| Fármacos con tautomería conocida | 12 | omeprazol, pirimetamina, metotrexato |
| Heterociclos simples | 10 | 4-piridona, imidazol, pirazol, triazoles |
| Ceto-enol no obvio | 10 | acetilacetona, malonaldehído, β-cetoésteres |

**El bosque ya está precalculado** (`p02_bosque_resultados.csv`).
Tu tarea es añadir tu semilla (2-piridona) y analizar el conjunto completo.

**📌 Espacio para figura** *(insertar aquí)*:
Mapa del bosque: panel 4×2 con un representante de cada familia,
su estructura 2D, y su población p₁ anotada.
Muy útil para dar contexto visual antes del análisis estadístico.


In [ ]:
# ── Cargar el bosque precalculado ────────────────────────────────────────────
# El archivo p02_bosque_resultados.csv contiene los resultados GFN2-xTB
# de las 40 moléculas. Fue generado con el script p02_expansion.py.

BOSQUE_CSV = 'p02_bosque_resultados.csv'

if os.path.exists(BOSQUE_CSV):
    df_bosque = pd.read_csv(BOSQUE_CSV)
    print(f'✓ Bosque cargado: {len(df_bosque)} moléculas')
    print(f'  Familias: {df_bosque["familia"].value_counts().to_dict()}')
else:
    print(f'⚠ No se encontró {BOSQUE_CSV}')
    print('  Opciones:')
    print('  a) Descarga el archivo del repositorio del curso')
    print('  b) Genera el bosque ejecutando: python p02_expansion.py')
    print('  c) Usa el bosque sintético de demostración (celdas siguientes)\n')

    # Bosque sintético para demostración pedagógica
    np.random.seed(42)
    n = 40
    familias_demo = (['base_nucleica']*8 + ['farmaco']*12 +
                     ['heterociclo']*10 + ['ceto_enol']*10)
    dG_T2 = np.concatenate([
        np.random.uniform(5,  25, 8),   # bases: moderado
        np.random.uniform(2,  15, 12),  # fármacos: disperso
        np.random.uniform(10, 35, 10),  # heterociclos: suele dominar
        np.random.uniform(1,  12, 10),  # ceto-enol: más mezclado
    ])
    pesos = np.exp(-dG_T2 / (R * T))
    pop1  = 100 / (1 + pesos)
    df_bosque = pd.DataFrame({
        'id'           : [f'mol_{i+1:02d}' for i in range(n)],
        'familia'      : familias_demo,
        'n_tautomeros' : np.random.randint(2, 8, n),
        'dG_T2_kJ'     : dG_T2.round(2),
        'pop_T1_pct'   : pop1.round(2),
        'pop_T2_pct'   : (100 - pop1).round(2),
        'dG_ref_kJ'    : np.where(np.random.rand(n) > 0.5, dG_T2 * np.random.uniform(0.8, 1.2, n), np.nan).round(2),
    })
    df_bosque.to_csv(BOSQUE_CSV, index=False)
    print(f'  ✓ Bosque sintético generado: {len(df_bosque)} moléculas')

print()
print(df_bosque.head())


In [ ]:
# ── Integrar la semilla al dataset ───────────────────────────────────────────
# El dataset final incluye el bosque + la semilla que calculaste.
# Esto es importante: NUNCA analices un conjunto sin tus propios datos en él.

# Recuperar valores de la semilla calculada
if len(df_semilla) >= 2:
    dG_T2_sem  = df_semilla['dG_kJ_mol'].iloc[1]
    pop1_sem   = df_semilla['porcentaje'].iloc[0]
    ntau_sem   = len(df_semilla)
else:
    dG_T2_sem, pop1_sem, ntau_sem = 6.3, 92.5, 2  # fallback

fila_semilla = pd.DataFrame([{
    'id'           : '2-piridona',
    'familia'      : 'heterociclo_ceto_enol',
    'n_tautomeros' : ntau_sem,
    'dG_T2_kJ'     : round(dG_T2_sem, 2),
    'pop_T1_pct'   : round(pop1_sem, 2),
    'pop_T2_pct'   : round(100 - pop1_sem, 2),
    'dG_ref_kJ'    : 6.3,  # Beak 1976
}])

df_final = pd.concat([df_bosque, fila_semilla], ignore_index=True)
df_final.to_csv('p02_dataset_final.csv', index=False)

print(f'Dataset final: {len(df_final)} moléculas (bosque + semilla)')
print(f'Semilla agregada: ΔG_T2={dG_T2_sem:.2f} kJ/mol, p1={pop1_sem:.1f}%')


---
## 📊 Análisis del bosque

### ¿Qué pregunta química guía el análisis?

> *¿Qué fracción de las moléculas farmacológicamente relevantes tiene
> un tautómero tan claramente dominante que podemos ignorar el resto?
> ¿Y en qué familias estructurales la incertidumbre tautómera es
> suficientemente grande como para invalidar un cálculo de propiedades?*


In [ ]:
# ── Estadística descriptiva del bosque ───────────────────────────────────────
# Antes de graficar, siempre explorar numéricamente el dataset.
# describe() calcula: conteo, media, desviación, min, cuartiles, max.

print('=== Estadística descriptiva del bosque ===')
print(df_final[['n_tautomeros', 'dG_T2_kJ', 'pop_T1_pct']].describe().round(2))

print(f'\nMoléculas con p1 > 99% (tautómero dominante claro): {(df_final["pop_T1_pct"]>99).sum()}')
print(f'Moléculas con p1 > 95%: {(df_final["pop_T1_pct"]>95).sum()}')
print(f'Moléculas con p1 < 80% (incertidumbre tautómera alta): {(df_final["pop_T1_pct"]<80).sum()}')
print(f'Moléculas con p1 < 60% (dos formas casi equipobladas): {(df_final["pop_T1_pct"]<60).sum()}')
print()

# Por familia
print('=== Media de p1 por familia ===')
print(df_final.groupby('familia')['pop_T1_pct'].agg(['mean','min','max']).round(1))


---
### 📈 Figura 1 — Distribución de poblaciones en el bosque

**Pregunta química:** ¿Qué fracción de las moléculas tiene un tautómero
dominante (p₁ > 95%)? ¿Y cuántas muestran incertidumbre tautómera
significativa (p₁ < 80%)?

El código de colores:
- 🟦 **Azul oscuro:** p₁ > 95% — seguro usar ese tautómero
- 🟦 **Azul medio:** 80–95% — predomina pero la forma alternativa es relevante
- 🩶 **Gris claro:** p₁ < 80% — incertidumbre tautómera alta


In [ ]:
# ── Figura 1: distribución de p1 para el bosque completo ─────────────────────
# Ordenamos por p1 ascendente para que el gráfico sea legible.
# Los colores comunican directamente la 'seguridad' de la asignación tautómera.

df_plot = df_final.sort_values('pop_T1_pct').reset_index(drop=True)

colores = ['#0F2747' if p > 95 else
           '#2A6F97' if p > 80 else
           '#AABBD0'
           for p in df_plot['pop_T1_pct']]

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(df_plot)), df_plot['pop_T1_pct'],
       color=colores, edgecolor='white', linewidth=0.5)

# Líneas de referencia
ax.axhline(95, color='#555', linestyle='--', linewidth=0.9, label='95% — umbral seguro')
ax.axhline(80, color='#888', linestyle=':',  linewidth=0.9, label='80% — umbral precaución')

# Anotar la semilla
idx_semilla = df_plot[df_plot['id'] == '2-piridona'].index
if len(idx_semilla) > 0:
    i = idx_semilla[0]
    ax.annotate('← semilla\n(2-piridona)',
                xy=(i, df_plot.loc[i, 'pop_T1_pct']),
                xytext=(i + 2, df_plot.loc[i, 'pop_T1_pct'] - 15),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.2),
                fontsize=8)

# Leyenda de colores
leyenda = [
    mpatches.Patch(color='#0F2747', label=f'p₁ > 95%  (N={sum(p>95 for p in df_final["pop_T1_pct"])})'),
    mpatches.Patch(color='#2A6F97', label=f'80% < p₁ ≤ 95%'),
    mpatches.Patch(color='#AABBD0', label=f'p₁ ≤ 80%  (N={sum(p<=80 for p in df_final["pop_T1_pct"])})'),
]
ax.legend(handles=leyenda + [plt.Line2D([0],[0],color='#555',ls='--',label='95%'),
                               plt.Line2D([0],[0],color='#888',ls=':',label='80%')],
          fontsize=8, loc='lower right')

ax.set_xlabel('Molécula (ordenada por p₁ ascendente)', fontsize=11)
ax.set_ylabel('Población del tautómero más estable p₁ (%)', fontsize=11)
ax.set_title('Distribución de poblaciones de Boltzmann — Bosque P02', fontsize=12)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig('p02_poblaciones_barra.pdf', dpi=150)
plt.show()
print('Guardada: p02_poblaciones_barra.pdf')


---
### 📈 Figura 2 — ΔG_T2 vs p₁, coloreada por familia

**Pregunta química:** ¿Existe una familia estructural que concentre
los casos de mayor incertidumbre tautómera?

Esta relación es **teóricamente perfecta** (curva de Boltzmann),
pero el scatter por familia revela cuáles familias exploran rangos
de ΔG más pequeños (y por tanto tienen más incertidumbre).


In [ ]:
# ── Figura 2: dispersión ΔG_T2 vs p1, coloreada por familia ─────────────────
# Esta figura es diagnóstica: una curva sigmoidal teórica + los puntos reales.
# Los puntos lejos de la curva indicarían errores en la extracción.

fig, ax = plt.subplots(figsize=(7, 5))

# Curva teórica de Boltzmann (para 2 formas: p1 = 1/(1+exp(-ΔG/RT)))
dG_range = np.linspace(0, 35, 300)
p1_teorica = 100 / (1 + np.exp(-dG_range / (R * T)))
ax.plot(dG_range, p1_teorica, 'k--', linewidth=1, alpha=0.4,
        label='Curva de Boltzmann (2 formas)')

# Puntos por familia
paleta = {'base_nucleica'       : '#E74C3C',
          'farmaco'             : '#3498DB',
          'heterociclo'         : '#27AE60',
          'ceto_enol'           : '#F39C12',
          'heterociclo_ceto_enol': '#8E44AD'}

for fam in df_final['familia'].unique():
    sub = df_final[df_final['familia'] == fam].dropna(subset=['dG_T2_kJ'])
    color = paleta.get(fam, '#95A5A6')
    ax.scatter(sub['dG_T2_kJ'], sub['pop_T1_pct'],
               label=fam, color=color,
               alpha=0.85, s=55, edgecolors='k', linewidths=0.4)

ax.set_xlabel(r'$\Delta G_{T_2}$ / kJ mol$^{-1}$', fontsize=11)
ax.set_ylabel(r'Población $p_1$ (%)', fontsize=11)
ax.set_title(r'$\Delta G_{T_2}$ vs $p_1$ por familia estructural', fontsize=12)
ax.legend(fontsize=8, loc='lower right')
ax.set_ylim(40, 102)
plt.tight_layout()
plt.savefig('p02_dG_vs_pop.pdf', dpi=150)
plt.show()
print('Guardada: p02_dG_vs_pop.pdf')


---
### 📈 Figura 3 — Validación: ΔG calculado vs experimental

**Pregunta química:** ¿Dónde falla GFN2-xTB respecto a datos
experimentales? Un **gráfico de paridad** (calculado vs experimental)
es la herramienta estándar para evaluar métodos computacionales.

- Puntos en la **diagonal (y=x)**: acuerdo perfecto
- Puntos por encima: el cálculo sobreestima ΔG
- Puntos por debajo: el cálculo subestima ΔG

El **MAE** (error absoluto medio) y el **r de Pearson** son las métricas
estándar de validación en quimioinformática.


In [ ]:
# ── Figura 3: paridad calculado vs experimental ───────────────────────────────
# Solo para moléculas con valor experimental disponible (dG_ref_kJ no nulo).
# El gráfico de paridad es la forma más directa de evaluar un método.

from scipy import stats as sp_stats

df_val = df_final.dropna(subset=['dG_ref_kJ', 'dG_T2_kJ']).copy()
print(f'Moléculas con datos experimentales: {len(df_val)}')

if len(df_val) >= 3:
    # Métricas de validación
    mae = np.mean(np.abs(df_val['dG_T2_kJ'] - df_val['dG_ref_kJ']))
    r, pval = sp_stats.pearsonr(df_val['dG_ref_kJ'], df_val['dG_T2_kJ'])

    fig, ax = plt.subplots(figsize=(5.5, 5.5))

    ax.scatter(df_val['dG_ref_kJ'], df_val['dG_T2_kJ'],
               edgecolors='k', linewidths=0.5, s=65,
               color='#2A6F97', zorder=5)

    # Diagonal de paridad
    lim = max(abs(df_val[['dG_ref_kJ', 'dG_T2_kJ']].values.flatten())) + 3
    ax.plot([-lim, lim], [-lim, lim], 'k--', linewidth=1, label='y = x')
    ax.set_xlim(-1, lim); ax.set_ylim(-1, lim)

    # Banda de ±3 kJ/mol (criterio de aceptación)
    ax.fill_between([-lim, lim], [-lim-3, lim-3], [-lim+3, lim+3],
                    alpha=0.1, color='#2A6F97', label='±3 kJ/mol')

    ax.set_xlabel(r'$\Delta G_{\mathrm{exp}}$ / kJ mol$^{-1}$', fontsize=11)
    ax.set_ylabel(r'$\Delta G_{\mathrm{calc}}$ (GFN2-xTB) / kJ mol$^{-1}$', fontsize=11)
    ax.set_title(f'Paridad | MAE = {mae:.1f} kJ/mol | r = {r:.3f}', fontsize=11)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig('p02_paridad.pdf', dpi=150)
    plt.show()

    print(f'MAE = {mae:.2f} kJ/mol')
    print(f'r de Pearson = {r:.3f} (p = {pval:.4f})')
    print(f'Criterio (<3 kJ/mol MAE): {"✓" if mae < 3 else "⚠"}')
    print('Guardada: p02_paridad.pdf')
else:
    print('Insuficientes datos con referencia experimental para el gráfico de paridad.')


---
## 🖥️ Tablero interactivo — Explorador del microespacio

El gráfico siguiente es completamente **interactivo**:
- **Pasa el cursor** sobre cada punto para ver el nombre, familia, ΔG y p₁
- **Haz clic** en la leyenda para mostrar/ocultar familias
- **Selecciona** una región con el rectángulo para hacer zoom
- **Doble clic** para restablecer la vista

Esta interactividad es esencial para identificar outliers y
casos interesantes en el dataset.


In [ ]:
# ── Tablero interactivo con Plotly ────────────────────────────────────────────
# Plotly genera HTML/JavaScript interactivo dentro del notebook.
# La clave es usar 'hover_data' para mostrar información útil al pasar el cursor.

if PLOTLY_OK:
    # Preparar datos
    df_plot_ly = df_final.dropna(subset=['dG_T2_kJ']).copy()
    df_plot_ly['es_semilla'] = df_plot_ly['id'] == '2-piridona'

    # Crear figura con subplots
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=[
            'ΔG_T2 vs p₁ por familia',
            'Distribución de p₁ (violín + scatter)'
        ],
        horizontal_spacing=0.12
    )

    paleta_px = px.colors.qualitative.Set2
    familias_unicas = df_plot_ly['familia'].unique()

    for i, fam in enumerate(familias_unicas):
        sub = df_plot_ly[df_plot_ly['familia'] == fam]
        color = paleta_px[i % len(paleta_px)]

        # Panel izquierdo: scatter ΔG vs p1
        fig.add_trace(
            go.Scatter(
                x=sub['dG_T2_kJ'],
                y=sub['pop_T1_pct'],
                mode='markers',
                name=fam,
                marker=dict(color=color, size=9,
                            line=dict(color='black', width=0.5)),
                customdata=sub[['id','n_tautomeros','pop_T2_pct']].values,
                hovertemplate=(
                    '<b>%{customdata[0]}</b><br>'
                    'ΔG_T2 = %{x:.1f} kJ/mol<br>'
                    'p₁ = %{y:.1f}%<br>'
                    'p₂ = %{customdata[2]:.1f}%<br>'
                    'N tautómeros = %{customdata[1]}<extra></extra>'
                ),
                showlegend=True,
                legendgroup=fam,
            ),
            row=1, col=1
        )

        # Panel derecho: violín + jitter por familia
        fig.add_trace(
            go.Violin(
                y=sub['pop_T1_pct'],
                x=[fam] * len(sub),
                name=fam,
                fillcolor=color,
                opacity=0.5,
                box_visible=True,
                meanline_visible=True,
                showlegend=False,
                legendgroup=fam,
                line_color='black',
                line_width=1,
            ),
            row=1, col=2
        )

    # Líneas de referencia
    for umbral, color_l, nombre_l in [(95,'#333','95%'), (80,'#666','80%')]:
        fig.add_hline(y=umbral, line_dash='dash', line_color=color_l,
                      annotation_text=nombre_l, row=1, col=1)

    fig.update_layout(
        title_text='<b>Tablero P02 · Microespacio de tautómeros</b>',
        title_font_size=16,
        height=480,
        template='plotly_white',
        legend=dict(title='Familia', font_size=11),
        violinmode='overlay'
    )
    fig.update_xaxes(title_text='ΔG_T2 (kJ/mol)', row=1, col=1)
    fig.update_yaxes(title_text='p₁ (%)', row=1, col=1)
    fig.update_xaxes(tickangle=20, row=1, col=2)

    fig.show()
    # Guardar también como HTML autocontenido
    fig.write_html('p02_tablero_interactivo.html')
    print('✓ Tablero guardado: p02_tablero_interactivo.html')

else:
    print('Plotly no disponible. Instala con: pip install plotly')


---
## 🔄 Revisión de hipótesis iniciales

Compara tus respuestas de las preguntas previas con los resultados obtenidos.


In [ ]:
# ── Mostrar respuestas iniciales para comparación ────────────────────────────
print('=== TUS RESPUESTAS INICIALES ===')
print('\nPregunta 1:', respuesta_1)
print('Pregunta 2:', respuesta_2)
print('Pregunta 3:', respuesta_3)

print('\n=== RESULTADOS OBTENIDOS ===')
print(f'Tautómero dominante: {df_semilla.iloc[0]["etiqueta"]} ({df_semilla.iloc[0]["porcentaje"]:.1f}%)')
print(f'ΔG calculado: {df_semilla["dG_kJ_mol"].iloc[1]:.2f} kJ/mol (ref: 6.3 kJ/mol)')
print(f'Familia con mayor dispersión: {df_final.groupby("familia")["pop_T1_pct"].mean().idxmin()}')


In [ ]:
reflexion_final = """
Escribe aquí tu reflexión:
- ¿Tus hipótesis fueron correctas?
- ¿Qué te sorprendió de los resultados?
- ¿Qué limitaciones identificas en el pipeline?
"""
print(reflexion_final)


---
## 📦 Entregables

| Tipo | Archivo | Descripción |
|---|---|---|
| **D1** | `2-piridona_tautomeros_resultados.csv` | G, ΔG y poblaciones de la semilla |
| **D2** | `p02_dataset_final.csv` | Bosque completo + semilla integrada |
| **D3** | `2-piridona_T01_xtb.out`, `T02_xtb.out` | Logs xTB de los dos tautómeros |
| **F1** | `p02_poblaciones_barra.pdf` | Distribución de p₁ en el bosque |
| **F2** | `p02_dG_vs_pop.pdf` | Dispersión ΔG_T2 vs p₁ por familia |
| **F3** | `p02_paridad.pdf` | Gráfico de paridad calc. vs exp. |
| **F4** | `p02_tablero_interactivo.html` | Tablero Plotly interactivo |
| **T1** | Reporte ≤ 2 páginas | Validación + F1 + F2 comentadas |
| **T2** | Respuestas discusión | ≤ 150 palabras por pregunta |


In [ ]:
# ── Checklist automático de entregables ──────────────────────────────────────
# pathlib.Path.exists() verifica la existencia del archivo.
# .stat().st_size > 0 confirma que no está vacío.

from pathlib import Path

entregables = [
    ('D1', f'{NOMBRE}_tautomeros_resultados.csv'),
    ('D2', 'p02_dataset_final.csv'),
    ('D3', f'{NOMBRE}_tautomeros/2-piridona_T01_xtb.out'),
    ('D3', f'{NOMBRE}_tautomeros/2-piridona_T02_xtb.out'),
    ('F1', 'p02_poblaciones_barra.pdf'),
    ('F2', 'p02_dG_vs_pop.pdf'),
    ('F3', 'p02_paridad.pdf'),
    ('F4', 'p02_tablero_interactivo.html'),
]

print('=== Checklist de entregables ===')
todos_ok = True
for tipo, archivo in entregables:
    p = Path(archivo)
    existe = p.exists()
    no_vacio = p.stat().st_size > 0 if existe else False
    estado = '✓' if (existe and no_vacio) else '✗'
    if not (existe and no_vacio):
        todos_ok = False
    print(f'  [{estado}] {tipo}: {archivo}')

print()
print('¡Todos los entregables listos!' if todos_ok else
      '⚠ Faltan algunos entregables — revisa las celdas anteriores.')


---
## ❓ Preguntas de discusión

Responde en el informe T2 (≤ 150 palabras por pregunta).


In [ ]:
# ── Preguntas de discusión ────────────────────────────────────────────────────
preguntas = {
    1: '(Comprensión) ¿Cuántos tautómeros encontró TautomerEnumerator para la '
       '2-piridona? ¿Son todos químicamente razonables? Si alguno no lo es, '
       'explica por qué el algoritmo lo genera a pesar de eso.',

    2: '(Comprensión) Identifica en el log xTB la línea que reporta G(298 K). '
       '¿Qué contribuciones energéticas suma xTB para obtenerla? '
       '¿Qué diferencia hay entre E_total y G(298 K)?',

    3: '(Análisis) Usa la ecuación de Boltzmann para calcular manualmente '
       'qué ΔG sería necesario para que el tautómero minoritario tuviera '
       'exactamente 1% de población. ¿A qué temperatura podría ser '
       'observable con técnica espectroscópica estándar?',

    4: '(Análisis) Del bosque, identifica la molécula con la distribución '
       'más "plana" (menor ΔG_T2). ¿Qué consecuencia tendría para un '
       'cálculo de docking ejecutado sobre el tautómero incorrecto?',

    5: '(Metodológico) El protocolo usa energías en vacío. ¿En qué dirección '
       'esperarías que se desplazara el equilibrio 2-piridona ⇌ '
       '2-hidroxipiridina al pasar de vacío a agua? ¿Y a ciclohexano?',

    6: '(Metodológico — SMARTS) RDKit enumera tautómeros con reglas SMARTS '
       'predefinidas. ¿Qué tipo de tautomería NO detectaría este algoritmo? '
       'Propón un ejemplo de transformación tautómera fuera del conjunto de '
       'reglas de Sitzmann.',
}

for num, texto in preguntas.items():
    print(f'P{num}. {texto}')
    print()


---
## 🔗 Conexión con el resto del manual

| Habilidad adquirida | Se usa en... |
|---|---|
| `TautomerEnumerator` + SMARTS | P04+: el tautómero dominante es el input correcto para todos los cálculos DFT |
| Distribución de Boltzmann sobre microespacio | P08 (conformaciones), P15 (rotámeros) |
| Fallback de energías simuladas | Patrón reutilizable en cualquier práctica donde xTB no esté disponible |
| Gráficos de paridad | Estándar de validación en todo el manual |
| Efecto del solvente (ausente aquí) | P28: PCM/SMD — cuantificará exactamente el desplazamiento del equilibrio |
| `p02_dataset_final.csv` | Input para P47 (exploración del espacio químico) |

---

> **⚠️ Nota para el docente / sugerencias de mejora al libro:**
>
> 1. **Diagrama 4-cuadrantes** (SMILES / SMARTS / SELFIES / estructura 2D):
>    insertar en la sección de cuadro conceptual para dar contexto visual.
>    Podría reutilizarse como figura de referencia en P01 también.
>
> 2. **Figura de mapa del bosque** (4×2 panel, un representante por familia):
>    insertar antes del bloque de análisis estadístico.
>
> 3. **Sección teórica del .tex:** considerar añadir un párrafo sobre SELFIES
>    en la sección de Marco Teórico, con referencia a Krenn et al. (2020)
>    `doi:10.1088/2632-2153/aba947`. El concepto es suficientemente nuevo
>    y con impacto directo en IA molecular para merecer mención explícita.
>
> 4. **Recurso adicional sugerido para el .tex:** tabla de tipos de tautomería
>    con sus patrones SMARTS y ejemplos de grupos funcionales que los exhiben.
>    No existe actualmente en el manual y sería un recurso de consulta rápida
>    muy valioso para los estudiantes.
>
> 5. **Extensión posible:** celda opcional de efecto de temperatura (recalcular
>    Boltzmann a T = 200, 298, 400, 600 K) — ya mencionada en extensiones del .tex.
